This notebook lists the Microsoft Excel (XLSX) items in the GIS portal
that are owned by the connected user, printing each item's title.
It then prepends a note about Microsoft Edge downloads to each item's description
(or sets the description to the note if there is none), skipping items that already have it.

In [1]:
import os
import sys


sys.path.insert(0, r'C:\Users\cpeak\repos\data-portal-tools')
from PortalConnector import PortalConnector

agol_user = os.getenv('AGOL_ADMIN_USERNAME')
agol_pw = os.getenv('AGOL_ADMIN_PW')
portal_conn = PortalConnector(
	portal_username=agol_user,
	portal_pw=agol_pw
 )


gis = portal_conn.gis
me = gis.users.me
print('Connected as:', me.username)

Connected as: gisadmin_PSRC


In [2]:
items = gis.content.search(query=f'owner:{me.username}', item_type='Microsoft Excel', max_items=10000)
print(f'Microsoft Excel items owned by {me.username}: {len(items)}')
print()
for item in sorted(items, key=lambda i: i.title.lower()):
    print(item.title)

c:\Users\cpeak\AppData\Local\anaconda3\envs\dpt2\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'psregcncl.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\cpeak\AppData\Local\anaconda3\envs\dpt2\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'psregcncl.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\cpeak\AppData\Local\anaconda3\envs\dpt2\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'psregcncl.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanc

Microsoft Excel items owned by gisadmin_PSRC: 95

ACS Data Profile: Demographic Characteristics, ACS 2019 1-Year Data_by_City
ACS Data Profile: Demographic Characteristics, ACS 2019 1-Year Data_by_State_County_MSA
ACS Data Profile: Demographic Characteristics, ACS 2020 5-Year Data_by_CDP
ACS Data Profile: Demographic Characteristics, ACS 2020 5-Year Data_by_City
ACS Data Profile: Demographic Characteristics, ACS 2020 5-Year Data_by_State_County_MSA
ACS Data Profile: Demographic Characteristics, ACS 2021 1-Year Data_by_City
ACS Data Profile: Demographic Characteristics, ACS 2021 1-Year Data_by_State_County_MSA
ACS Data Profile: Demographic Characteristics, ACS 2021 5-Year Data_by_CDP
ACS Data Profile: Demographic Characteristics, ACS 2021 5-Year Data_by_City
ACS Data Profile: Demographic Characteristics, ACS 2021 5-Year Data_by_State_County_MSA
ACS Data Profile: Demographic Characteristics, ACS 2022 1-Year Data_by_City
ACS Data Profile: Demographic Characteristics, ACS 2022 1-Year Data_

In [4]:
# Prepend the Edge download note to each item's description.
# DRY_RUN = True only reports what would change; set it to False to apply.
import html
import re

DRY_RUN = False

EDGE_NOTE = (
    '<p><strong>Using Microsoft Edge?</strong> If the download returns a 404 error, '
    'go to Edge Settings → Downloads (<code>edge://settings/downloads</code>) and turn off '
    '<em>"Open Office files in the browser,"</em> then try again.</p>'
)


def normalize(text):
    # Reduce HTML to comparable plain text, so edits AGOL makes when saving a
    # description (entities like &quot; or &rarr;, &nbsp;, tag/attribute
    # rewrites, smart quotes, line breaks) don't hide an existing note.
    text = re.sub(r'<[^>]*>', ' ', text or '')
    text = html.unescape(text)
    text = text.translate(str.maketrans({'“': '"', '”': '"', '‘': "'", '’': "'"}))
    return re.sub(r'\s+', ' ', text).strip()


EDGE_NOTE_NORMALIZED = normalize(EDGE_NOTE)

updated, skipped, failed = [], [], []
for item in sorted(items, key=lambda i: i.title.lower()):
    try:
        current = item.description or ''
        if EDGE_NOTE_NORMALIZED in normalize(current):
            skipped.append(item.title)
            continue
        new_description = EDGE_NOTE + current
        if DRY_RUN:
            print(f'[dry run] {item.title}: would {"prepend note" if current else "set description"}')
        else:
            if not item.update(item_properties={'description': new_description}):
                raise RuntimeError('item.update returned False')
            print(f'updated {item.title}')
        updated.append(item.title)
    except Exception as e:
        failed.append((item.title, str(e)[:150]))

print()
print(f'{"would update" if DRY_RUN else "updated"}: {len(updated)}')
print(f'already had note: {len(skipped)}')
print(f'failed: {len(failed)}')
for title, msg in failed:
    print(f'  {title}: {msg}')

c:\Users\cpeak\AppData\Local\anaconda3\envs\dpt2\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'psregcncl.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\cpeak\AppData\Local\anaconda3\envs\dpt2\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'psregcncl.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\cpeak\AppData\Local\anaconda3\envs\dpt2\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'psregcncl.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanc

updated ACS Data Profile: Demographic Characteristics, ACS 2019 1-Year Data_by_City


c:\Users\cpeak\AppData\Local\anaconda3\envs\dpt2\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'psregcncl.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\cpeak\AppData\Local\anaconda3\envs\dpt2\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'psregcncl.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\cpeak\AppData\Local\anaconda3\envs\dpt2\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'psregcncl.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanc

updated ACS Data Profile: Demographic Characteristics, ACS 2019 1-Year Data_by_State_County_MSA


c:\Users\cpeak\AppData\Local\anaconda3\envs\dpt2\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'psregcncl.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\cpeak\AppData\Local\anaconda3\envs\dpt2\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'psregcncl.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\cpeak\AppData\Local\anaconda3\envs\dpt2\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'psregcncl.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanc

...

updated Decennial Data Profile: General Population and Housing Characteristics, 2020_by_CDP


c:\Users\cpeak\AppData\Local\anaconda3\envs\dpt2\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'psregcncl.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\cpeak\AppData\Local\anaconda3\envs\dpt2\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'psregcncl.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\cpeak\AppData\Local\anaconda3\envs\dpt2\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'psregcncl.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanc

updated Decennial Data Profile: General Population and Housing Characteristics, 2020_by_City


c:\Users\cpeak\AppData\Local\anaconda3\envs\dpt2\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'psregcncl.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\cpeak\AppData\Local\anaconda3\envs\dpt2\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'psregcncl.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\cpeak\AppData\Local\anaconda3\envs\dpt2\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'psregcncl.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanc

updated Decennial Data Profile: General Population and Housing Characteristics, 2020_by_State_County_MSA

updated: 95
already had note: 0
failed: 0
